<a href="https://colab.research.google.com/github/Bo-fromLA/ML-EDA-projects/blob/main/final_cal_Housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV

In [ ]:
df1=pd.read_csv('/content/california_df1.csv')
df1.head(2)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,6k_income_groups,6q_house_values,7q_income_qcut,2lat_lon_house_val
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0,0,0,0,1
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0,0,0,0,1


In [ ]:
y=df1['median_house_value']
X=df1.drop('median_house_value',axis=1)

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X.shape   # (17000, 12)

(17000, 12)

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('ordinal', OrdinalEncoder(), ['6q_house_values', '7q_income_qcut']),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['2lat_lon_house_val', '6k_income_groups'])
])

# RandomForestRegressor pipeline
forest_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=1, n_jobs=-1))
])

# XGBRegressor pipeline
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=1, n_jobs=-1))
])

In [ ]:
# Grid search parameters for RandomForestRegressor
param_grid_forest = {
    'regressor__n_estimators': [100, 150],
    'regressor__max_depth': [10, 15, 20],
    'regressor__min_samples_split': [100, 150, 175]
}

# GridSearchCV for RandomForestRegressor
grid_search_forest = GridSearchCV(forest_model, param_grid_forest, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search_forest.fit(x_train, y_train)

print("Best parameters for RandomForestRegressor: ", grid_search_forest.best_params_)
print("Best score for RandomForestRegressor: ", -grid_search_forest.best_score_)

# Best parameters for RandomForestRegressor:  {'regressor__max_depth': 10, 'regressor__min_samples_split': 30, 'regressor__n_estimators': 100}
# Best score for RandomForestRegressor:  17940.591894488938

Best parameters for RandomForestRegressor:  {'regressor__max_depth': 10, 'regressor__min_samples_split': 150, 'regressor__n_estimators': 150}
Best score for RandomForestRegressor:  17932.70374771497


In [ ]:
# Evaluate RandomForestRegressor on test set
forest_test_score = grid_search_forest.best_estimator_.score(x_test, y_test)
forest_pred = grid_search_forest.best_estimator_.predict(x_test)

print('\nRandom Forest score on test set: ', forest_test_score)
print('Random Forest MAE on test set: ', mean_absolute_error(y_test, forest_pred))



Random Forest score on test set:  0.9379017692400374
Random Forest MAE on test set:  19083.51508291744


In [ ]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
x_test_preprocessed = preprocessor.transform(x_test)

# Pipeline with XGBRegressor
# xgb_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', XGBRegressor(n_estimators=2000, random_state=1, learning_rate=0.03, max_depth=10))
#])
xgb_manual_model=XGBRegressor(n_estimators=1000, random_state=1, learning_rate=0.01, max_depth=25, subsample=0.4, colsample_bytree=0.8)
xgb_manual_model.fit(x_train_preprocessed, y_train,
              early_stopping_rounds = 5,
              eval_set = [(x_test_preprocessed, y_test)],
              verbose = False)

xgb_manual_model.score(x_test_preprocessed, y_test)
xgb_pred = xgb_manual_model.predict(x_test_preprocessed)
print('xgb r2: ', xgb_manual_model.score(x_test_preprocessed, y_test))
print('xgb MAE: ',mean_absolute_error(xgb_pred, y_test))

/usr/local/lib/python3.10/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


xgb r2:  0.9377267194545499
xgb MAE:  19060.426362591912


In [ ]:
df2=df1.copy()
X_og = df2.drop('median_house_value', axis=1)
y_og = df2['median_house_value']

original_test = cross_val_score(xgb_manual_model, X_og, y_og, cv=5, scoring='neg_mean_absolute_error')
original_test.mean()
print('Original MAE: ', -original_test.mean())

Original MAE:  -17992.037284237133


In [ ]:
data=pd.read_csv('/content/sample_data/california_housing_test.csv')
data.head(2)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0


In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=6, random_state=1)

kmeans.fit(data[['median_income']])
data['6k_income_groups'] = kmeans.labels_

# Here we are just sorting the Kmeans labels by their ascending order

cluster_mins = data.groupby('6k_income_groups')['median_income'].min()
sorted_clusters = cluster_mins.sort_values().index

new_labels = {old_label: new_label for new_label, old_label in enumerate(sorted_clusters)}
data['6k_income_groups'] = data['6k_income_groups'].map(new_labels)


/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [ ]:
data['6q_house_values'] = pd.qcut(data['median_house_value'], q=6, labels=False)
data['7q_income_qcut'] = pd.qcut(data['median_income'], q=7, labels=False)

kmeans_house_value_data = KMeans(n_clusters=2, random_state=1).fit(data[['longitude', 'latitude', 'median_house_value']])
data['2lat_lon_house_val']=kmeans_house_value_data.labels_
data.shape

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


(3000, 13)

In [ ]:
X_data = data.drop('median_house_value', axis=1)
y_data = data['median_house_value']

x_train_preprocessed = preprocessor.fit_transform(x_train)
x_test_preprocessed = preprocessor.transform(X_data)


In [ ]:
X_data_preprocessed = preprocessor.transform(X_data)

# XGB SEEMS TO BE THE WINNER FOLLOWED VERY CLOSELY BY THE RANDOM FOREST REGRESSOR

In [ ]:
#XGB

predictions=xgb_manual_model.predict(X_data_preprocessed)

print('xgb r2: ', xgb_manual_model.score(x_test_preprocessed, y_data))
print('xgb MAE: ',mean_absolute_error(predictions, y_data))

# xgb r2:  0.9372525167453598
# xgb MAE:  18338.7579140625

xgb r2:  0.9372525167453598
xgb MAE:  18338.7579140625


In [ ]:
# FOREST

predictions_forest=grid_search_forest.best_estimator_.predict(X_data)

print('Random Forest MAE: ',mean_absolute_error(predictions_forest, y_data))
print('Random Forest R^2: ', r2_score(y_data, predictions_forest))

# Random Forest MAE:  18392.110770170853
# Random Forest R^2:  0.9368286159150157

Random Forest MAE:  18392.110770170853
Random Forest R^2:  0.9368286159150157


# TRYING THE CORRELATION COLUMNS ONLY

In [ ]:
X_data_corr = X_data.drop(columns=['longitude', 'population', 'total_bedrooms'])
X_data_corr.shape


(3000, 9)

In [ ]:
X_data_corr_preprocessed = preprocessor.transform(X_data_corr)

In [ ]:
corr_preds= xgb_manual_model.predict(X_data_corr_preprocessed)

print('final xgb r2: ', xgb_manual_model.score(X_data_corr_preprocessed, y_data))
print('final xgb MAE: ',mean_absolute_error(corr_preds, y_data))



final xgb r2:  0.9372525167453598
final xgb MAE:  18338.7579140625


In [ ]:
for_preds=grid_search_forest.best_estimator_.predict(X_data_corr)

print('final Random Forest MAE: ',mean_absolute_error(for_preds, y_data))
print('final Random Forest R^2: ', r2_score(y_data, for_preds))

final Random Forest MAE:  18392.110770170853
final Random Forest R^2:  0.9368286159150157


In [ ]:
# Grid search parameters for XGBRegressor
param_grid_xgb = {
    'regressor__n_estimators': [1000, 1500, 2000],
    'regressor__max_depth': [5, 10, 15],
    'regressor__learning_rate': [0.01, 0.03, 0.05, 0.1]
}

# GridSearchCV for XGBRegressor
grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, error_score='raise')
#grid_search_xgb.set_params(regressor__early_stopping_rounds=20, regressor__eval_set=[(x_test_preprocessed, y_test)], regressor__verbose=False)
grid_search_xgb.fit(x_train_preprocessed, y_train)

#grid_search_xgb.fit(x_train, y_train, **{'regressor__eval_set': [(x_test, y_test)], 'regressor__early_stopping_rounds': 5, 'regressor__verbose': False})
#grid_search_xgb.fit(x_train, y_train, regressor__eval_set=[(x_test, y_test)], regressor__early_stopping_rounds=5, regressor__verbose=False)

print("Best parameters for XGBRegressor: ", grid_search_xgb.best_params_)
print("Best score for XGBRegressor: ", -grid_search_xgb.best_score_)

In [ ]:
# Evaluate on test set
xgb_test_score = grid_search_xgb.best_estimator_.score(x_test, y_test)
xgb_pred = grid_search_xgb.best_estimator_.predict(x_test)

print('\nXGBoost score on test set: ', xgb_test_score)
print('XGBoost MAE on test set: ', mean_absolute_error(y_test, xgb_pred))
print('XGBoost R^2 on test set: ', r2_score(y_test, xgb_pred))

In [ ]:
# param_grid_forest = {
#     'regressor__n_estimators': [100, 200, 300],
#     'regressor__max_depth': [None, 10, 20],
#     'regressor__min_samples_split': [2, 5, 10]
# }

param_grid_forest = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 15, 20],
    'regressor__min_samples_split': [25, 30, 40]
}

# GridSearchCV for RandomForestRegressor
grid_search_forest = GridSearchCV(forest_model, param_grid_forest, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, error_score='raise')
grid_search_forest.fit(x_train, y_train)

# Best parameters and best scores
print("Best parameters for RandomForestRegressor: ", grid_search_forest.best_params_)
print("Best score for RandomForestRegressor: ", -grid_search_forest.best_score_)
# Best parameters for RandomForestRegressor:  {'regressor__max_depth': 10, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}
# Best score for RandomForestRegressor:  -17941.269140878012

# Best parameters for RandomForestRegressor:  {'regressor__max_depth': 10, 'regressor__min_samples_split': 15, 'regressor__n_estimators': 100}
# Best score for RandomForestRegressor:  17941.164599031916

# Best parameters for RandomForestRegressor:  {'regressor__max_depth': 10, 'regressor__min_samples_split': 25, 'regressor__n_estimators': 100}
# Best score for RandomForestRegressor:  17940.87418132446

Best parameters for RandomForestRegressor:  {'regressor__max_depth': 10, 'regressor__min_samples_split': 25, 'regressor__n_estimators': 100}
Best score for RandomForestRegressor:  17940.874181324452


In [ ]:
# Evaluate on test set
print('\nEvaluating on test set - Random Forest:')

forest_test_score = grid_search_forest.best_estimator_.score(x_test, y_test)
forest_pred = grid_search_forest.best_estimator_.predict(x_test)

print('\nRandom Forest score on test set: ', forest_test_score)
print('Random Forest MAE on test set: ', mean_absolute_error(y_test, forest_pred))
print('Random Forest R^2 on test set: ', r2_score(y_test, forest_pred))


# Evaluating on test set - Random Forest:

# Random Forest score on test set:  0.9377901018984651
# Random Forest MAE on test set:  19090.29465615457
# Random Forest R^2 on test set:  0.9377901018984651



Evaluating on test set - Random Forest:

Random Forest score on test set:  0.9377901018984651
Random Forest MAE on test set:  19090.29465615457
Random Forest R^2 on test set:  0.9377901018984651


In [ ]:
param_grid_xgb = {
    'regressor__n_estimators': [1000, 2000],
    'regressor__max_depth': [5, 10, 15],
    'regressor__learning_rate': [0.01, 0.03, 0.07]
}

# GridSearchCV for XGBRegressor

grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, error_score='raise')
grid_search_xgb.fit(x_train, y_train, regressor__eval_set=[(x_test, y_test)], regressor__early_stopping_rounds=5, regressor__verbose=False)

print("Best parameters for XGBRegressor: ", grid_search_xgb.best_params_)
print("Best score for XGBRegressor: ", -grid_search_xgb.best_score_)

# Best parameters for XGBRegressor:  {'regressor__learning_rate': 0.01, 'regressor__max_depth': 5, 'regressor__n_estimators': 1000}
# Best score for XGBRegressor:  17943.535904756434

XGBoostError: [03:16:11] /workspace/src/data/iterative_dmatrix.cc:93: Check failed: ref->Info().num_col_ == n_features (10 vs. 12) : Invalid ref DMatrix, different number of features.
Stack trace:
  [bt] (0) /usr/local/lib/python3.10/dist-packages/xgboost/lib/libxgboost.so(+0x3effba) [0x7ce70b811fba]
  [bt] (1) /usr/local/lib/python3.10/dist-packages/xgboost/lib/libxgboost.so(+0x3f35ce) [0x7ce70b8155ce]
  [bt] (2) /usr/local/lib/python3.10/dist-packages/xgboost/lib/libxgboost.so(+0x3f5b01) [0x7ce70b817b01]
  [bt] (3) /usr/local/lib/python3.10/dist-packages/xgboost/lib/libxgboost.so(+0x3f8858) [0x7ce70b81a858]
  [bt] (4) /usr/local/lib/python3.10/dist-packages/xgboost/lib/libxgboost.so(+0x3a2a07) [0x7ce70b7c4a07]
  [bt] (5) /usr/local/lib/python3.10/dist-packages/xgboost/lib/libxgboost.so(XGQuantileDMatrixCreateFromCallback+0x2b0) [0x7ce70b587c40]
  [bt] (6) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x7ce74ccfae2e]
  [bt] (7) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x7ce74ccf7493]

In [ ]:
# Evaluate on test set
print('\nEvaluating on test set XGB Regressor:')
xgb_test_score = grid_search_xgb.best_estimator_.score(x_test, y_test)
xgb_pred = grid_search_xgb.best_estimator_.predict(x_test)

print('\nXGBoost score on test set: ', xgb_test_score)
print('XGBoost MAE on test set: ', mean_absolute_error(y_test, xgb_pred))
print('XGBoost R^2 on test set: ', r2_score(y_test, xgb_pred))

# Evaluating on test set XGB Regressor:

# XGBoost score on test set:  0.9377570297679122
# XGBoost MAE on test set:  19096.4408203125
# XGBoost R^2 on test set:  0.9377570297679122


Evaluating on test set XGB Regressor:

XGBoost score on test set:  0.9377570297679122
XGBoost MAE on test set:  19096.4408203125
XGBoost R^2 on test set:  0.9377570297679122
